In [62]:
from sklearn import *

### Key Points from Lecture

1. **Topics Covered**:
   - Pipelines, parameter optimization, and handling of test, validation, and training data.

2. **Exam and Practice Exam**:
   - Practice exam will be available on Blackboard, but it may have vague phrasing and other issues, unlike the final polished exam.

3. **Model Quality**:
   - A good model accurately predicts outcomes for new data and generalizes well.

4. **Data Leakage**:
   - Occurs when test data inadvertently influences training, leading to overly optimistic results.
   - **Types**:
     - **Target Leakage**: Using information that would not be available at prediction time, especially problematic in time-series data.
     - **Train-Test Contamination**: Allowing the test set to influence model training.

5. **Handling Data Leakage**:
   - Split data into training and testing before any processing.
   - Use pipelines to standardize and avoid accidental leakage.

6. **Model Evaluation and Cross-Validation**:
   - Use K-Fold Cross-Validation to split data and obtain reliable performance metrics.
   - Stratified sampling can ensure each class is proportionately represented in training and testing.

7. **Parameter Tuning and Hyperparameters**:
   - Grid search and cross-validation help find optimal hyperparameters.
   - Carefully manage randomness (using fixed seeds) to ensure consistency and prevent overfitting.

8. **Pipeline Structure**:
   - Combine transformers (e.g., scalers) and estimators (e.g., classifiers) in a pipeline for organized processing.
   - Pipelines prevent test data leakage by encapsulating the steps into one workflow.

9. **Multiple Classifiers in a Pipeline**:
   - Try different classifiers in the same pipeline to compare and select the best model.

10. **Hyperparameter Optimization Techniques**:
    - **Grid Search CV**: Tries all parameter combinations, effective but computationally heavy.
    - **Randomized Search CV**: Samples parameter settings randomly, faster than grid search.

11. **Final Model Selection**:
    - The ultimate goal is to report model accuracy reliably, not necessarily to find the "perfect" model.

12. **Recommendations**:
    - Keep test data completely separate until the final evaluation.
    - Use available tools (e.g., `scikit-learn` pipelines) to avoid common pitfalls in model development.

13. **Portfolio Assignments**:
    - Focus on feature engineering, model evaluation, ensemble learning, and hyperparameter optimization. Assignment will be available by the end of the day.

### Important Tips
- Always split data before any transformation.
- Avoid using test data during any training phase.
- Use cross-validation and pipelines to manage workflows and prevent data leakage.
- Randomized search is less exhaustive but more efficient than grid search for hyperparameter tuning.
- Stratified sampling helps maintain class distribution across splits.

# The Goal

You want to give your boss more than just a "good" model. You want to give him the set of "best" hyperparameters to use, as well as an indication of how performant (*e.g.* accurate) the resulting model will be.

# What makes a good model?

A model is good if it can make accurate predictions for unseen data.

# What is data leakage?

Data leakage happens if data is available for training that is not available for prediction. This can be split into 2 classes:

- Target leakage:
  + Time series data, make sure you don't have data from the future available when doing training
  + Make sure your features are not derived from targets. For example, a dataset with target `has_pneumonia` and a feature `is_treated_for_pneumonia`. Most likely the feature is a result of the target. 
 
- Train-test contamination
  + See following code examples

In [63]:
# Slightly more complex dataset than used in the lecture
data = datasets.load_digits()
raw_X = data.data
y = data.target
print(raw_X.shape, y.shape)

(1797, 64) (1797,)


In [64]:
# Resubstitution: No way to tell how accurate my model is for unseen data! (Also, overfitting)
scaler = preprocessing.StandardScaler()
X = scaler.fit_transform(raw_X)
model = ensemble.RandomForestClassifier()
model.fit(X, y)
yhat = model.predict(X)
score = metrics.f1_score(y, yhat, average='micro')
print(score)

1.0


***

In [65]:
# First scaling/preprocessing, then splitting. Already better, but here the preprocessing has
# access to (testing) data that is not available when making predictions. In this specific
# example it knows something about the distribution of the unseen data
scaler = preprocessing.StandardScaler()
X = scaler.fit_transform(raw_X)
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y)
model = ensemble.RandomForestClassifier()
model.fit(X_train, y_train)
yhat = model.predict(X_test)
score = metrics.f1_score(y_test, yhat, average='micro')
print(score)

0.9777777777777777


***

In [66]:
# This is pretty decent, but it's tedious because you need to manually
# feed data through your preprocessing and model, and make sure you don't forget one.

X_train, X_test, y_train, y_test = model_selection.train_test_split(raw_X, y)

scaler = preprocessing.StandardScaler()
X = scaler.fit_transform(X_train)
model = ensemble.RandomForestClassifier()

model.fit(X_train, y_train)
yhat = model.predict(X_test)
score = metrics.f1_score(y_test, yhat, average='micro')
print(score)

0.9755555555555555


***

In [67]:
# Pipelines make it easy to chain Transformers and Estimators. The problem here
# is that I split my data every time I want to try different hyperparameters, which
# means I can just keep trying untill I find a split I like.

X_train, X_test, y_train, y_test = model_selection.train_test_split(raw_X, y)

model = pipeline.make_pipeline(  # See also pipeline.Pipeline
    preprocessing.StandardScaler(),
    ensemble.RandomForestClassifier(
        n_estimators=25,
        criterion='entropy',
        max_depth=250,
        min_samples_split=2,
        min_samples_leaf=2,
        min_weight_fraction_leaf=0.0,
        max_features=None,
        max_leaf_nodes=None,
        min_impurity_decrease=0.0,
    ),
)

model.fit(X_train, y_train)
yhat = model.predict(X_test)
score = metrics.f1_score(y_test, yhat, average='micro')
print(score)

0.9555555555555556


***

In [68]:
# Here the problem is that I cannot use the printed score to tune my 
# hyperparameters, because then the hyperparameters will eventually encode my 
# testing data

In [69]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(raw_X, y)

In [70]:
model = pipeline.make_pipeline(  # See also pipeline.Pipeline
    preprocessing.StandardScaler(),
    ensemble.RandomForestClassifier(
        n_estimators=100,
        criterion='entropy',
        max_depth=250,
        min_samples_split=2,
        min_samples_leaf=2,
        min_weight_fraction_leaf=0.0,
        max_features="log2",
        max_leaf_nodes=None,
        min_impurity_decrease=0.0,
    ),
)

model.fit(X_train, y_train)
yhat = model.predict(X_test)
score = metrics.f1_score(y_test, yhat, average='micro')
print(score)

0.9755555555555555


***

# (Stratified) K-fold cross validation and hyperparameter optimisation

K-fold cross validation: split the *training*\* data into K folds. Pick one of the folds as validation set, and train a model on the others. Validate the performance of that model using the validation set. Do this for all K combinations and average the resulting scores.

Hyperparameters determine the behaviour of your model, for example the strength of the regularisation or the criterion used for determining splits in a tree. You want to optimize these to find the best possible model.

Stratified splits: Let's say you have 100 samples, 20 positive and 80 negative. If you then randomly draw 20 samples (for example for your test set), the chance of drawing 20 negative samples is non-zero. The solution is making sure you draw 20% of your positive samples and 20% of your negative samples, so that the distribution of classes in your test set is the same as in your total data. Usually, sklearn does the right thing.

\* Your test data is in a vault, so you can't use that

In [71]:
# Generate your train/test data, and put the test data in a vault.
X_train, X_test, y_train, y_test = model_selection.train_test_split(raw_X, y)

In [72]:
model = pipeline.make_pipeline(  # See also pipeline.Pipeline
    preprocessing.StandardScaler(),
    ensemble.RandomForestClassifier(),
)

print(model.get_params())  # This shows the parameters we can play with. Not all are useful.

{'memory': None, 'steps': [('standardscaler', StandardScaler()), ('randomforestclassifier', RandomForestClassifier())], 'verbose': False, 'standardscaler': StandardScaler(), 'randomforestclassifier': RandomForestClassifier(), 'standardscaler__copy': True, 'standardscaler__with_mean': True, 'standardscaler__with_std': True, 'randomforestclassifier__bootstrap': True, 'randomforestclassifier__ccp_alpha': 0.0, 'randomforestclassifier__class_weight': None, 'randomforestclassifier__criterion': 'gini', 'randomforestclassifier__max_depth': None, 'randomforestclassifier__max_features': 'sqrt', 'randomforestclassifier__max_leaf_nodes': None, 'randomforestclassifier__max_samples': None, 'randomforestclassifier__min_impurity_decrease': 0.0, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__min_samples_split': 2, 'randomforestclassifier__min_weight_fraction_leaf': 0.0, 'randomforestclassifier__n_estimators': 100, 'randomforestclassifier__n_jobs': None, 'randomforestclassifier_

In [73]:
# This does a hyperparameter optimization. The objective function is determined by the 
# `scoring` argument, and the search space for the hyperparameters by the `param_grid`.
# The GridSearch algorithm just tries all possible combinations and returns the best.

# See also: RandomizedSearchCV, Halving(Grid|Randomized)SearchCV
gridsearch = model_selection.GridSearchCV(
    estimator=model,
    param_grid={
        'randomforestclassifier__n_estimators': [25, 50, 75, 100, 250],
        'randomforestclassifier__criterion': ['gini', 'entropy'],
        'randomforestclassifier__max_depth': [None, 10, 25],
        'randomforestclassifier__min_samples_split': [2],
        'randomforestclassifier__min_samples_leaf': [1],
        'randomforestclassifier__min_weight_fraction_leaf': [0.0],
        'randomforestclassifier__max_features': [None, "sqrt", "log2"],
        'randomforestclassifier__max_leaf_nodes': [None],
        'randomforestclassifier__min_impurity_decrease': [0.0],
    },
    cv=5,
    scoring=metrics.make_scorer(metrics.f1_score, average='micro'),
    n_jobs=-1,
)


gridsearch.fit(X_train, y_train)
best = gridsearch.best_estimator_
print(gridsearch.best_params_)
print(gridsearch.best_score_)
yhat = best.predict(X_test)
score = metrics.f1_score(y_test, yhat, average='micro')
print(score)

{'randomforestclassifier__criterion': 'entropy', 'randomforestclassifier__max_depth': None, 'randomforestclassifier__max_features': 'log2', 'randomforestclassifier__max_leaf_nodes': None, 'randomforestclassifier__min_impurity_decrease': 0.0, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__min_samples_split': 2, 'randomforestclassifier__min_weight_fraction_leaf': 0.0, 'randomforestclassifier__n_estimators': 100}
0.9777227041167562
0.9822222222222222


In [74]:
# Different classifier algorithms? Ugly hack incoming!
# Here, we explore the idea "what if the type of algorithm I use is a hyperparameter?"

model = pipeline.Pipeline([
    ('scale', preprocessing.StandardScaler()),
    ('clf', dummy.DummyClassifier()),
])

print(model.get_params())  # Note that 'clf' is one of the available parameters!

{'memory': None, 'steps': [('scale', StandardScaler()), ('clf', DummyClassifier())], 'verbose': False, 'scale': StandardScaler(), 'clf': DummyClassifier(), 'scale__copy': True, 'scale__with_mean': True, 'scale__with_std': True, 'clf__constant': None, 'clf__random_state': None, 'clf__strategy': 'prior'}


In [75]:
param_grid = [
    {
        'clf': [ensemble.RandomForestClassifier()],
        'clf__n_estimators': [25, 50, 75, 100, 250],
        'clf__criterion': ['gini', 'entropy'],
        'clf__max_depth': [None, 10, 25],
        'clf__min_samples_split': [2],
        'clf__min_samples_leaf': [1],
        'clf__min_weight_fraction_leaf': [0.0],
        'clf__max_features': [None, "sqrt", "log2"],
        'clf__max_leaf_nodes': [None],
        'clf__min_impurity_decrease': [0.0],
    },
    {
        'clf': [linear_model.LogisticRegression()],
        'clf__penalty': ['l1', 'l2', 'elasticnet'],
        'clf__C': [1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3],
        'clf__class_weight': ['balanced'],
        'clf__l1_ratio': [0.5],
        'clf__solver': ['saga'],
    }
    
]

gridsearch = model_selection.GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring=metrics.make_scorer(metrics.f1_score, average='micro'),
    n_jobs=-1
)


gridsearch.fit(X_train, y_train)
best = gridsearch.best_estimator_
print(gridsearch.best_params_)
print(gridsearch.best_score_)
yhat = best.predict(X_test)
score = metrics.f1_score(y_test, yhat, average='micro')
print(score)

/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio paramet

/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not co

/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warni

/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr

{'clf': RandomForestClassifier(criterion='entropy', max_depth=25, max_features='log2',
                       n_estimators=250), 'clf__criterion': 'entropy', 'clf__max_depth': 25, 'clf__max_features': 'log2', 'clf__max_leaf_nodes': None, 'clf__min_impurity_decrease': 0.0, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2, 'clf__min_weight_fraction_leaf': 0.0, 'clf__n_estimators': 250}
0.9762357152691725
0.9844444444444445
